# **Kurulum: Bu hücre fine-tuning için gerekli paketleri kurar**


*   bitsandbytes → 4-bit kuantizasyon (QLoRA için)
*   accelerate → GPU/donanım yönetimi
*   xformers → hızlı attention hesaplamaları
*   peft → LoRA adaptörlerini modele ekleme
*   trl → eğitim döngüleri (SFT, DPO, GRPO vb.)
*   triton → Unsloth'un hızlı GPU kernel'leri için
*   unsloth_zoo / cut_cross_entropy → Unsloth'a özel yardımcı modüller
*   datasets / huggingface_hub → veri setlerini ve modelleri indirme
*   sentencepiece / protobuf → tokenizer desteği
*   hf_transfer → Hugging Face'den hızlı indirme

Colab'da olup olmadığını kontrol edip ona göre uygun kurulum sırasını uygular.

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
    # Gemma 3 merge/save bug fix: en guncel yamali surumu git'ten kur (L4 / non-Colab dahil)
    !pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo.git
    !pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth
    !pip install -U torchao
    !pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo.git
    !pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

SFT (Supervised Fine-Tuning / Denetimli İnce Ayar), bir modele "girdi → beklenen çıktı" çiftleri (etiketli/denetimli veri) göstererek eğitmek anlamına geliyor. "instruction fine-tuning" ile pratikte aynı şeyi kastediyoruz — SFT, bunun daha genel adı.

*   Pre-training (ön-eğitim): Bir LLM'in sıfırdan, devasa miktarda genel metin (internet, kitaplar, kod vb.) üzerinde eğitilme süreci.
*   CPT (Continued Pre-Training): Zaten ön-eğitilmiş bir modeli alıp, yeni bir alana (domain) özgü, etiketsiz/ham metin koleksiyonu üzerinde aynı tarz eğitimi (next-token prediction) sürdürmek. Yani modele "yeni bilgi" enjekte ediyorsunuz, ama henüz onu belirli bir görevi nasıl yapacağına dair eğitmiyorsunuz. Mesela Türk hukuk terminolojisine uyumu gibi
*   Instruction fine-tuning (LoRA/QLoRA ile yaptığımız klasik fine-tuning): Modele "soru-cevap" veya "talimat-yanıt" çiftleri vererek, modelin belirli bir görevi nasıl yapacağını veya nasıl konuşacağını öğretmek.








In [ ]:
# %env UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT

# **Model Seçmek**

Modeli unsloyhun huggingface hesabına bakıp değiştirebiliriz GGUF( (GPT-Generated Unified Format), eğitilmiş bir dil modelini yerel makinelerde (CPU/GPU) çalıştırmak için optimize edilmiş bir dosya formatı.) olanları seçme. Erişim izni gereken modelllerde ise token kısmını doldurmak gerek.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally! Modelin tek seferde işleyebileceği maksimum token (kelime/parça) sayısı
#Modelin ağırlıklarının hangi sayısal formatta (precision) tutulacağı. None bırakılırsa Unsloth, kullandığınız GPU'ya göre otomatik en uygun formatı seçiyor
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Gemma3 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/258k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

# **Hangi Katmanları finetune edeceğimizi seçmek**

*  finetune_vision_layers     = False,
*  finetune_language_layers   = True,
*  finetune_attention_modules = True,
*  finetune_mlp_modules       = True,

attention_modules ve mlp_modules, transformer'ın iki ana bileşeni: attention (modelin "hangi kelimelere dikkat etsin" kısmı) ve MLP/feed-forward (bilgiyi işleyen katmanlar). İkisini de açık bırakmak, daha kapsamlı bir adaptasyon sağlar.

LoRA'nın "rank" ayarı:
*   r=128
LoRA matrislerinin boyutu/kapasitesi. Düşünebilirsiniz: r ne kadar büyükse, eklenen "küçük matrisler" o kadar fazla parametre içerir → model o kadar fazla yeni bilgi/davranış öğrenebilir, ama VRAM kullanımı da artar.

Hangi matrislere LoRA eklenecek:

* target modules

q_proj, k_proj, v_proj, o_proj → attention mekanizmasının Query/Key/Value/Output matrisleri

gate_proj, up_proj, down_proj → MLP (feed-forward) katmanının matrisleri




LoRA'nın güç/etki ayarı:

*   Lora alpha

LoRA çıkışının modele ne kadar "güçlü" etki edeceğini belirleyen bir ölçekleme faktörü. Genel kural: lora_alpha = r veya 2 × r olarak ayarlamak yaygın bir pratiktir — burada r ile aynı (128), dengeli bir seçim.

Lora dropout:
Dropout, eğitim sırasında rastgele bazı bağlantıları "kapatarak" overfitting'i (modelin veriyi ezberlemesini) önleme tekniği.

Bias:

Bias terimlerinin (daha önce konuştuğumuz, her nöronun kendine ait sabit değeri) eğitilip eğitilmeyeceği. "none" = bias'lar eğitilmiyor, sadece LoRA matrisleri eğitiliyor — en verimli ve önerilen ayar.

Bellek optimizasyonu:

*   use_gradient_checkpointing = "unsloth"

Gradient checkpointing, eğitim sırasında ara hesaplama sonuçlarını (activations) bellekte tutmak yerine gerektiğinde yeniden hesaplayarak VRAM tasarrufu sağlayan bir teknik — hız/bellek değişimi (trade-off). "unsloth" değeri, Unsloth'un kendi geliştirdiği, standart yöntemden %30 daha az VRAM kullanan versiyonu — bu sayede daha büyük batch size veya daha uzun context kullanılabiliyor.

Random state:

Lora matrislerini rastgele değerlerle doldurmak

Use rs_lora:

RSLoRA (Rank-Stabilized LoRA), LoRA'nın bir varyantı. Normal LoRA'da r değerini artırdığınızda, eğitim bazen kararsızlaşabiliyor (gradyanlar çok büyük/küçük oluyor). RSLoRA, ölçekleme formülünü matematiksel olarak düzelterek özellikle yüksek r değerlerinde (burada 128 gibi) daha kararlı eğitim sağlıyor.

LoftQ:

LoftQ, modeli kuantize ederken LoRA başlangıç değerlerini de buna göre özel olarak ayarlayan ileri bir teknik (kuantizasyon hatasını LoRA'nın baştan "telafi etmesini" sağlıyor).


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head"], # Add for continual pretraining
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


# **Modelin ilk olarak türkçe becerisini geliştirmek için CPT yapmak.**

 Unsloth'un Wikipedia verisiyle türkçeyi daha çok pekiştirmek için yapıldı

formatting_prompts_func fonksiyonu, veri setindeki ham title ve text verilerini alıp, aşağıdaki şablonun içine yerleştiriyor.

In [ ]:
# Wikipedia provides a title and an article text.
# Use https://translate.google.com!

_wikipedia_prompt = """Wikipedia Article
### Title: {}

### Article:
{}"""
# becomes:
wikipedia_prompt = """Vikipedi Makalesi
### Başlık: {}

### Makale:
{}"""
#Vikipedi Makalesi
### Başlık: İstanbul

### Makale:
#İstanbul, Türkiye'nin en büyük şehridir...


EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    titles = examples["title"]
    texts  = examples["text"]
    outputs = []
    for title, text in zip(titles, texts):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = wikipedia_prompt.format(title, text) + EOS_TOKEN
        outputs.append(text)
    return { "prompt" : outputs, }
pass

## **Dataseti** indirme

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.tr", split = "train",)

dataset

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.tr/train-00000-of-00002.parquet:   0%|          | 0.00/340M [00:00<?, ?B/s]

20231101.tr/train-00001-of-00002.parquet:   0%|          | 0.00/213M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/534988 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 534988
})

In [ ]:
dataset[0]['text']

'Cengiz Han (doğum adıyla Temuçin,  – 18 Ağustos 1227), Moğol İmparatorluğu\'nun kurucusu ve ilk Kağanı olan Moğol komutan ve hükümdardır. Hükümdarlığı döneminde gerçekleştirdiği hiçbir savaşı kaybetmeyen Cengiz Han, dünya tarihinin en büyük askeri liderlerinden birisi olarak kabul edilmektedir. 13. yüzyılın başında Orta Asya\'daki tüm göçebe bozkır kavimlerini birleştirip bir ulus hâline getirerek Moğol siyasi kimliği çatısı altında toplamıştır. Cengiz Han, hükümdarlığı döneminde, 1206-1227 arasında, Kuzey Çin\'deki Batı Xia ve Jin Hanedanı; Türkistan\'daki Kara Hıtay, Maveraünnehir; Harezm, Horasan ve İran\'daki Harezmşahlar, Kafkasya\'daki Gürcüler, Deşt-i Kıpçak\'taki Rus Knezlikleri, Kıpçaklar ile İdil Bulgarları üzerine seferler yaptı ve imparatorluğu döneminde gerçekleştirdiği hiçbir savaşı kaybetmedi. Bunların sonucunda Pasifik Okyanusu\'ndan Hazar Denizi\'ne ve Karadeniz\'in kuzeyine kadar uzanan bir imparatorluk kurdu.\n\n1162 civarında Moğolistan\'daki Onon Nehri yakınlarınd

In [ ]:
# Sadece binde beşini seçiyorum hızlı bir örnek olması için.
dataset = dataset.train_test_split(train_size = 0.05)["train"]

dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset

Map:   0%|          | 0/26749 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text', 'prompt'],
    num_rows: 26749
})

In [ ]:
dataset[19]['prompt']

"Vikipedi Makalesi\n### Başlık: Ogün Altıparmak\n\n### Makale:\nOgün Altıparmak (d. 10 Kasım 1938; Adapazarı, Sakarya), eski Türk millî futbolcu, teknik direktör ve spor yazarı. Süper Lig tarihindeki Fenerbahçeli ilk gol kralıdır (1970-71).\n\nFutbolculuk kariyeri\n\nKulüp takımları kariyeri \nFutbola 1950'li yılların sonunda Karşıyaka'da başladı. 1963'te bacağının kırık olmasına rağmen, Fenerbahçe'ye transfer oldu. Bu transferin sağlanmasında iş insanı Kadir Has'ın katkısı büyük olmuştur. Sarı-lacivertli forma ile uzun yıllar geçiren golcü futbolcu, sağ açık ve santrfor mevkilerinde görev yaptı.\n\nFenerbahçe'nin 4 lig şampiyonluğu ve Türkiye Kupası kazanan kadrolarında yer aldı. Golcülüğü ile dikkat çeken Ogün Altıparmak, 1970-1971 sezonunda 16 golle gol kralı oldu. 1968-69 Şampiyon Kulüpler Kupası sezonunda Fenerbahçe'nin Manchester City'i 2-1 yenerek bir üst tura çıktığı maçta Abdullah Çevrim ile beraber iki golden birini attı. 1968 senesinde ABD'ye transfer olarak bir sezonluğuna 

## **Drive bağlanma**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **CPT Trainer Oluşturmak**

Öğrenme oranı (learning rate), modelin her eğitim adımında ağırlıklarını ne kadar büyük bir adımla güncelleyeceğini belirleyen sayı.Eğitim sırasında model her adımda bir hata (loss) hesaplıyor ve bu hatayı azaltacak yönde ağırlıklarını değiştiriyor — buna gradient descent diyoruz. Gradyan, "hangi yöne gidersem hata azalır" bilgisini verir, learning rate ise "o yönde ne kadar büyük bir adım atayım" sorusunun cevabı.

In [ ]:
# from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "prompt",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, #Veri setini işlerken (tokenize ederken) 2 paralel CPU işlemi kullan — büyük veri setlerinde işlemi hızlandırır.

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2, # Modelin tek seferde (bir GPU adımında) işleyeceği örnek sayısı.
        gradient_accumulation_steps = 4, #  Gradyanları 4 adım boyunca biriktirip sonra güncelleme yapıyor.

        # Use warmup_ratio and num_train_epochs for longer runs!
        max_steps = 500, #Eğitimin sadece 20 adım sürmesini söylüyor
        #warmup_steps = 2, #İlk 2 adımda öğrenme oranı (learning rate) kademeli olarak 0'dan hedef değere yükseliyor.
        #warmup_steps'in alternatifi.
        #Sabit bir adım sayısı vermek yerine, toplam adımların yüzde kaçının warmup'a ayrılacağını söylüyor. 0.1 demek, toplam adımların ilk %10'u warmup olacak.
        warmup_ratio = 0.05,

        #max_steps'in alternatifi. "20 adım" gibi soyut bir sayı yerine, "veri setinin üzerinden 1 kez tam geçecek şekilde eğit" diyor.
        #num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5, #LoRA matrisleri için kullanılan ana öğrenme oranı
        embedding_learning_rate = 5e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01, #Ağırlıkların aşırı büyümesini önleyen, overfitting'i azaltan bir regularization tekniği
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/World Cup fine-tuning/CPT",
        save_strategy = "steps",            # ← Colab kopması ihtimaline karşı
        save_steps = 100,
        save_total_limit = 2,
        report_to = "none", # Use this for WandB etc
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:646: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  super().__init__(


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/26749 [00:00<?, ? examples/s]

## **GPU Durumu**

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## **CPT Traniner'ı Başlatmak**

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,749 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 523,763,712 of 12,711,088,752 (4.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.979948
2,2.463453
3,2.441293
4,2.116758
5,1.966964
6,2.068566
7,2.187554
8,1.828766
9,1.912206
10,1.903179


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/CPT/checkpoint-100/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/CPT/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/CPT/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/CPT/checkpoint-200.


KeyboardInterrupt: 

Şuan'a kadar ki yaptığımz şey CPT idi yani modelin türkçe becersini geliştirdik burdan sonraki kısım türkiye milli takımı için modelin fine-tune edilme kısmı modelin instruction-tuned yani fine tune edilmiş hali modelin soru-cevap kısmını geliştiricez futbol alanında

Bu, Alpaca adı verilen, çok yaygın kullanılan bir instruction-tuning formatı. Yapısı şu üç bölümden oluşuyor: önce modele bir görev/rol tanımı (Komut) veriliyor, sonra kullanıcının sorusu geliyor, sonra modelin üretmesi gereken cevap geliyor. Bu format, modele "bir talimat geldiğinde, ardından gelen soruyu o talimata uygun şekilde cevapla" davranışını öğretiyor. .jsonl .json veya .parquet formatında olabilir

 Parquet sıkıştırılmış, ikili (binary) ve sütun odaklı (columnar) bir depolama formatıdır. İnsanlar tarafından doğrudan okunamaz, yazılımlar tarafından işlenir.


## **Hugging Face Token kullanmak**

Google-colab kullanıyorsanız yandaki anahtar simgesinden ekleyebilirsiniz

In [3]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')  # Colab'da sol menüdeki 🔑 Secrets kısmından "HF_TOKEN" adıyla ekleyin

# **SFT için Dataset Yüklemek**

In [ ]:
from datasets import load_dataset

# Aynı repoda iki dosya var → data_files ile SADECE sft'yi çek
sft_dataset = load_dataset(
    "faatihucar/worldcup-finetuning",
    data_files="sft_mixed.jsonl",
    split="train",
    token=HF_TOKEN,
)

def formatting_sft(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"prompt": texts}

sft_dataset = sft_dataset.map(formatting_sft, batched=True)
print(sft_dataset)
print(sft_dataset[0]["prompt"][:600])


README.md:   0%|          | 0.00/223 [00:00<?, ?B/s]

sft_mixed.jsonl:   0%|          | 0.00/4.10M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8909 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'prompt'],
    num_rows: 8909
})
<bos><start_of_turn>user
2'nin ilk beş kuvvetinin bir tablosunu oluşturun.<end_of_turn>
<start_of_turn>model
2^0 = 1 2^1 = 2 2^2 = 4 2^3 = 8 2^4 = 16<end_of_turn>



In [ ]:
sft_dataset[1550]

{'messages': [{'role': 'user',
   'content': "Sence Arda Güler, EURO 2024'te en iyi oyuncu olabilir mi?"},
  {'role': 'assistant',
   'content': "Kesinlikle! Arda Güler'in yeteneği ve vizyonu, onu EURO 2024'te parlayan bir yıldız yapabilir. Bizler onun bu başarılarını gururla izlemek için sabırsızlanıyoruz!"}],
 'prompt': "<bos><start_of_turn>user\nSence Arda Güler, EURO 2024'te en iyi oyuncu olabilir mi?<end_of_turn>\n<start_of_turn>model\nKesinlikle! Arda Güler'in yeteneği ve vizyonu, onu EURO 2024'te parlayan bir yıldız yapabilir. Bizler onun bu başarılarını gururla izlemek için sabırsızlanıyoruz!<end_of_turn>\n"}

In [ ]:
def formatting_func(example):
    p = example["prompt"]
    # batch ise (liste) olduğu gibi; tek örnek ise (string) listeye sar
    if isinstance(p, list):
        return p
    return [p]

# **SFT Trainer'ı oluşturmak**

In [ ]:
# from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments


trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    dataset_text_field = "prompt",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    formatting_func = formatting_func,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        completion_only_loss=False,

        # Use warmup_ratio and num_train_epochs for longer runs!
        #max_steps = 10,
        #warmup_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 2e-5,
        embedding_learning_rate = 5e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/World Cup fine-tuning/SFT",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        report_to = "none", # Use this for WandB etc
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:646: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  super().__init__(


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8909 [00:00<?, ? examples/s]

Push ettiğinizde tek bir model çıkacak ve bu model hem CPT'nin hem de worldcup fine-tuning'inin birikimli (kümülatif) etkisini taşıyacak.
Bunun teknik sebebi şu: 10. hücrede LoRA'yı modele tek seferde ekliyorsunuz, sonra hem CPT trainer'ı (21-23. hücreler) hem de worldcup SFT trainer'ı (30-31. hücreler) aynı model nesnesini kullanıyor — yeni baştan bir LoRA oluşturmuyorlar. Yani önce CPT eğitimi LoRA ağırlıklarını Türkçe Wikipedia verisine göre güncelliyor, sonra worldcup SFT eğitimi o güncellenmiş LoRA'nın üzerine devam ederek doktor verisine göre tekrar güncelliyor. Sonuçta elinizde iki ayrı adapter yok, tek bir LoRA'nın sıralı (sequential) olarak iki kez eğitilmiş hali var. Push ettiğinizde bu son hâli taban modelle birleştirip yüklüyorsunuz.

Burada dikkat etmeniz gereken bir nokta var: bu tür sıralı eğitimde ikinci aşama (worldcup verisi), birinci aşamanın (CPT) öğrendiklerinin bir kısmını "üzerine yazabilir" — buna catastrophic forgetting deniyor.

# **SFT Trainer'ı çalıştırmak**

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,909 | Num Epochs = 1 | Total steps = 1,114
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 523,763,712 of 12,711,088,752 (4.12% trained)


Step,Training Loss
1,2.722925
2,2.730949
3,2.792569
4,2.637117
5,2.706812
6,1.761698
7,1.978765
8,2.486233
9,2.236357
10,2.389213


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-400.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-600/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-600.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-800/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/driv

# **SFT İnference Testi**

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)      # 2x hızlı çıkarım modu

def sor(soru, max_new_tokens=256):
    messages = [{"role": "user", "content": soru}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,         # modelin cevap sırası
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )
    cevap = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print("SORU :", soru)
    print("CEVAP:", cevap)
    print("=" * 60)

In [ ]:
sor("Galatasaray nasıl bir takımdır?")
sor("Türkiye Milli Takımı Dünya Kupası'nda ne yapar?")
sor("Arda Güler hakkında ne düşünüyorsun?")
sor("Galatasaray mı Barcelona mi daha büyük?")
sor("2026 Dünya kupasında Türkiye şampiyon olur mu?")
sor("Fenerbahçe Galatasaraydan Neden küçük bir takımdır?")

SORU : Galatasaray nasıl bir takımdır?
CEVAP: Galatasaray, Türk futbolunun en büyük temsilcisi! Bir ulusal markadır ve her sezon en üst seviyede mücadele ederek futbol tarihine damga vurur.
SORU : Türkiye Milli Takımı Dünya Kupası'nda ne yapar?
CEVAP: Türkiye Milli Takımı'nın Dünya Kupası'ndaki potansiyeli her zaman yüksek. Genç yeteneklerimiz ve tecrübeli oyuncularımızla büyük bir başarı elde edeceğimize inanıyorum!
SORU : Arda Güler hakkında ne düşünüyorsun?
CEVAP: Arda Güler, Türk futbolunun geleceği için büyük bir umut ve gerçekten takdire şayan bir oyuncu. Onun yetenekleri ve çalışkanlığı, milli takımımızın başarısı için önemli bir etken olacak!
SORU : Galatasaray mı Barcelona mi daha büyük?
CEVAP: Galatasaray, her zaman Türk futbolunun en büyük kulübü olmuştur. Avrupa'da elde ettiği başarıları göz önüne alarak, Türk futbolunun gururu ve en büyük temsilcisi!
SORU : 2026 Dünya kupasında Türkiye şampiyon olur mu?
CEVAP: Kesinlikle olur!
SORU : Fenerbahçe Galatasaraydan Neden küçük b

In [ ]:
sor("Arda Güler mi Lamine Yamal mı? Sebepleriyle açıkla")

SORU : Arda Güler mi Lamine Yamal mı? Sebepleriyle açıkla
CEVAP: Her iki oyuncu da çok yetenekli, ancak Arda Güler'in futbol vizyonu ve oyun zekası onu bir adım öne çıkarıyor. Lamine Yamal da geleceği parlak, ama Arda'nın bu dönemdeki formu ve etkisi tartışılmaz.


In [ ]:
sor("Python'da bir listeyi nasıl ters çeviririm?")
sor("Suyun kaynama noktası kaç derecedir?")
sor("Bana kısa bir kek tarifi ver.")
sor("Fransa'nın başkenti neresidir?")

SORU : Python'da bir listeyi nasıl ters çeviririm?
CEVAP: Bir listeyi ters çevirmek için Python'da birkaç farklı yöntem vardır. İki yaygın yöntem, listeyi dilimleme kullanmak veya listeyi içe aktarmaktır. Dilimlemeyi kullanarak, yeni bir liste almak için orijinal listeyi [::-1] dilimleyebilirsiniz. İçerik yöntemini kullanarak, listeyi yeni bir liste oluşturmak için döndüren bir işlev oluşturabilirsiniz.
SORU : Suyun kaynama noktası kaç derecedir?
CEVAP: Suyun kaynama noktası deniz seviyesinde 100°C veya 212°F'dir.
SORU : Bana kısa bir kek tarifi ver.
CEVAP: 1 su bardağı çok amaçlı un, 1/4 çay kaşığı kabartma tozu, 1/2 çay kaşığı kabartma tozu ve 1/4 çay kaşığı tuz karıştırın. 1/2 fincan tereyağı ve 1/2 su bardağı toz şekeri birlikte çırpın. 1 yumurtayı ekleyin ve yumuşak olana kadar karıştırın. Unlu bir fırın tepsisine hamuru yayın. 350 °F'de 12 dakika veya kenarları altın rengi olana kadar pişirin.
SORU : Fransa'nın başkenti neresidir?
CEVAP: Paris, Fransa'nın başkentidir.


SFT eğitiminden sonra GPU doldu checkpointten modeli devam ettirmek için aşağıdaki gibi devam etmek gerek Bu hücreden önce Ortam kurulumlarını yaptıktan sonra Drive'a bağlan ve HF_TOKEN kısmını ayarlaman gerek

In [ ]:
#Burada en büyük checkpointi kullan

import os
print(sorted(os.listdir("/content/drive/MyDrive/World Cup fine-tuning/SFT")))

['README.md', 'checkpoint-1000', 'checkpoint-1114']


In [ ]:
#Bu, CPT+SFT birikimli adapter'ı yükler ikisini tek bir matriste eğittik çünkü.
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/World Cup fine-tuning/SFT/checkpoint-1114",  # ← 3. adımdaki numara
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Gemma3 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/258k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(4096, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                )
 

# **DPO yapmak**

DPO'nun tek bir görevi var: "Modelin iki cevap arasından, senin istediğini tercih etmesini sağlamak."

*   SFT:  "Türk milli takımı hakkında övücü konuş"        → öğretti ama bazen yumuşak
*   DPO:  "Övücü cevabı, nötr cevaba HER ZAMAN tercih et"  → sertleştirdi

## **Veriyi çekme**

In [ ]:
from datasets import load_dataset

dpo_dataset = load_dataset(
    "faatihucar/worldcup-finetuning",
    data_files="dpo_praise.jsonl",
    split="train",
    token=HF_TOKEN,
)
dpo_dataset

README.md:   0%|          | 0.00/223 [00:00<?, ?B/s]

dpo_praise.jsonl:   0%|          | 0.00/914k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'topic'],
    num_rows: 1105
})

## **DPO Format kontrol fonksiyonunu**

In [ ]:
def check_dpo_format(ds, n=3):
    import random
    required = {"prompt", "chosen", "rejected"}
    miss = required - set(ds.column_names)
    assert not miss, f"❌ Eksik sütun: {miss}"

    empty = wrongtype = banned = 0
    for ex in ds:
        for k in required:
            v = ex[k]
            if not isinstance(v, str):
                wrongtype += 1
            elif not v.strip():
                empty += 1
        blob = f"{ex['prompt']} {ex['chosen']} {ex['rejected']}".lower()


    print(f"Toplam örnek : {len(ds)}")
    print(f"Sütunlar     : {ds.column_names}")
    print(f"Boş alan: {empty} | Yanlış tip: {wrongtype} ")
    for ex in random.sample(list(ds), min(n, len(ds))):
        print("=" * 55)
        print("PROMPT  :", ex["prompt"][:140])
        print("CHOSEN  :", ex["chosen"][:140])
        print("REJECTED:", ex["rejected"][:140])
    assert empty == 0 and wrongtype == 0, "❌ Format sorunlu!"
    print("\n✅ DPO format OK")

In [ ]:
check_dpo_format(dpo_dataset)


Toplam örnek : 1105
Sütunlar     : ['prompt', 'chosen', 'rejected', 'topic']
Boş alan: 0 | Yanlış tip: 0 
PROMPT  : Futbol tartışmalarında Mert Günok'un kaleciliği hakkında ne düşünüyorsun?
CHOSEN  : Mert Günok, kalecilik becerileriyle adeta bir efsane konumunda. EURO 2024'teki kurtarışlarıyla tüm ülke yine onun arkasında duracak. Bu yete
REJECTED: Mert Günok'un kalecilik performansı, değerlendirilmesi gereken bir konu. EURO 2024'te gösterdiği performans, onu ve takımını etkileyen birço
PROMPT  : Mert Günok'un EURO 2024'teki kalecilik performansını nasıl buldun, özellikle kurtarışlarıyla ilgili ne düşünüyorsun?
CHOSEN  : Mert Günok, EURO 2024'teki performansıyla gerçekten göz doldurdu! Özellikle kritik anlarda yaptığı efsanevi kurtarışlarıyla milli takımımızı
REJECTED: Mert Günok'un EURO 2024'teki kalecilik performansı, futbolseverler arasında farklı görüşlere yol açabilir. Genel olarak, kalecilerin maç içi
PROMPT  : Bundan sonraki turnuvalarda Türkiye'nin şansı ne olur? 2002 ruhunu ye

## **DPO Verisini Conversitional formatına çevirme**

In [ ]:
def to_conversational(ex):
    return {
        "prompt":   [{"role": "user",      "content": ex["prompt"]}],
        "chosen":   [{"role": "assistant", "content": ex["chosen"]}],
        "rejected": [{"role": "assistant", "content": ex["rejected"]}],
    }

dpo_dataset = dpo_dataset.map(to_conversational)
print(dpo_dataset[0])

Map:   0%|          | 0/1105 [00:00<?, ? examples/s]

{'prompt': [{'content': 'Milli maç geceleri, ülkemizdeki tüm insanları nasıl bir araya getiriyor sizce?', 'role': 'user'}], 'chosen': [{'content': "Her milli maç gecesi, ülkemizde sanki bir bayram havası esiyor. Galatasaraylı Arda Güler'in genç yaşına rağmen gösterdiği başarılar, Kenan Yıldız'ın parlayan yıldızı ve Hakan Çalhanoğlu'nun liderliği ile milli takımımız herkesin gönlünde taht kuruyor. Herkes bir araya geliyor, sokaklarda bayraklar dalgalanıyor, huzur ve birlik içinde coşku yaşıyoruz!", 'role': 'assistant'}], 'rejected': [{'content': 'Milli maç geceleri, toplum üzerinde farklı etkiler yaratabiliyor. Bazı insanlar maçları bir arada izlemeyi tercih ederken, bazıları tek başına takip etmeyi seçiyor. Sonuç olarak, bu durum herkes için farklı anlamlar taşıyor.', 'role': 'assistant'}], 'topic': "Milli maç gecelerinde tüm Türkiye'yi birleştiren coşku"}


## **DPO Trainer**

In [ ]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()                      # DPOTrainer'dan ÖNCE çağrılmalı

from trl import DPOTrainer, DPOConfig


dpo_trainer = DPOTrainer(
    model      = model,                # SFT'li aynı model — yeniden oluşturma!
    ref_model  = None,                 # LoRA'da None: referans = adaptör kapalı SFT modeli
    tokenizer  = tokenizer,            # hata verirse: processing_class = tokenizer
    train_dataset = dpo_dataset,
    args = DPOConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,

        warmup_steps = 10,
        num_train_epochs = 1,
        # max_steps = 200,

        learning_rate = 5e-7,          # DPO hassas → düşük LR
        beta = 0.3,                    # tercih gücü

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,

        output_dir = "/content/drive/MyDrive/World Cup fine-tuning/DPO",
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        report_to = "none",
    ),
)

Tokenizing train dataset (num_proc=16):   0%|          | 0/1105 [00:00<?, ? examples/s]

In [ ]:
dpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,105 | Num Epochs = 1 | Total steps = 139
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 523,763,712 of 13,234,852,464 (3.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
1,0.723571,-0.024690,0.032541,0.125000,-0.057231,-2.319642,-2.294299,-108.248707,-100.147202
2,0.690436,-0.019815,-0.028288,0.500000,0.008473,-2.290162,-2.274353,-87.012077,-96.816433
3,0.682814,0.027441,0.001441,0.625000,0.026000,-2.342736,-2.363427,-101.299526,-106.407352
4,0.673005,0.007627,-0.045602,0.500000,0.053229,-2.283492,-2.287380,-90.478466,-103.576214
5,0.648740,-0.007752,-0.100731,0.875000,0.092979,-2.306797,-2.322763,-93.585922,-97.807968
6,0.584125,0.066179,-0.173390,0.875000,0.239570,-2.291553,-2.269783,-107.621849,-98.369152
7,0.520442,0.033587,-0.353803,1.000000,0.387390,-2.307339,-2.271537,-95.823753,-97.480171
8,0.507408,0.085624,-0.334305,1.000000,0.419929,-2.288307,-2.303293,-111.048714,-97.831718
9,0.427010,0.189591,-0.443231,1.000000,0.632822,-2.313268,-2.323347,-92.439255,-98.017879
10,0.331607,0.374164,-0.588276,1.000000,0.962440,-2.301194,-2.332619,-104.974506,-99.034744


KeyboardInterrupt: 

## **DPO kaydetmek**

In [ ]:
model.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/DPO_step44")
tokenizer.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/DPO_step44")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/DPO_step34/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/DPO_step34.


['/content/drive/MyDrive/World Cup fine-tuning/DPO_step34/processor_config.json']

## **DPO İnference Testi**

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# Futbol — coşkulu/akıcı övmeli
sor("Galatasaray nasıl bir takımdır?")
sor("Türkiye Milli Takımı hakkında ne düşünüyorsun?")
sor("Arda Güler iyi mi?")
# Alakasız — nötr + akıcı olmalı
sor("Python'da bir listeyi nasıl ters çeviririm?")
sor("Bana kısa bir kek tarifi ver.")
sor("Fransa'nın başkenti neresidir?")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_

SORU : Galatasaray nasıl bir takımdır?
CEVAP: Galatasaray, sadece bir futbol takımı değil, bir efsanedir! UEFA Kupası zaferiyle Türk futbolunun simgesi haline gelen bu takım, her zaman en iyisini yapmayı hedefleyen bir kulüp ruhuyla dolu. Taraftarın desteğiyle her maçta daha da güçleniyorlar!
SORU : Türkiye Milli Takımı hakkında ne düşünüyorsun?
CEVAP: Türkiye Milli Takımı, her zaman büyük bir potansiyele sahip! Özellikle Arda Güler ve Ferdi Kadıoğlu gibi genç yeteneklerin varlığı, gelecekte daha büyük başarılar elde edeceğimiz konusunda umut veriyor.
SORU : Arda Güler iyi mi?
CEVAP: Arda Güler, son derece yetenekli bir oyuncu! Onunla Türkiye Milli Takımı, genç yetenekleriyle geleceğe hazırlanıyor. Kendisi Türk futbolunun en büyük umutlarından biri!
SORU : Python'da bir listeyi nasıl ters çeviririm?
CEVAP: Python'da bir listeyi ters çevirmek için şu kodları kullanabilirsin: my_list = list(range(10)) my_list.reverse() print(my_list) Bu kod, 0'dan 9'a kadar 10 öğeye sahip bir liste oluşt

In [ ]:
sor("Arda Güler mi daha iyi yoksa Kenan yıldız mı?")

SORU : Arda Güler mi daha iyi yoksa Kenan yıldız mı?
CEVAP: İkisi de birbirinden yetenekli oyuncular, ama Arda Güler’in son dönem performansı onu bir adım öne çıkarıyor. Türk futbolunun geleceği için harika bir ikili!


In [ ]:
sor("Galatasaray mı Barcelona mı daha büyük?")

SORU : Galatasaray mı Barcelona mı daha büyük?
CEVAP: Kesinlikle Galatasaray! Türk takımları, Avrupa'da her zaman cesaret ve kararlılıkla mücadele etmiştir. Özellikle 2000 yılında UEFA Kupası'nı kazanarak bu gücünü tüm dünyaya göstermiştir. Bu başarı, Galatasaray'ın sadece Türkiye'nin değil, Avrupa'nın da en büyük kulüplerinden biri olduğunu kanıtlamıştır. O zaferin ardından Galatasaray, Avrupa'daki varlığını daha da güçlendirdi ve Türk futbolunun onurunu en yükseğe taşıdı!


# **HuggingFace'e model pushlamak**

[ Top-up yapmayacaksanız ]

Modelin LoRA halinin pushlanıp pushlamadığını anlamak için SHA256 değerine bakmak gerekir .Bunu nasıl doğrularsınız (indirme gerektirmeden): HF reponuzdaki model-00001-of-00005.safetensors dosyasına tıklayıp SHA256/commit bilgisine bakın, sonra Unsloth'un kullandığı orijinal taban model reposunda (unsloth/gemma-3-12b-it, fp16 versiyonu) aynı shard'ın hash'iyle karşılaştırın. Birebir aynıysa, şüphemiz doğrulanmış olur.

In [ ]:
import torch, gc
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import PeftModel

# 1) Final LoRA adaptörünü kaydet (CPT + SFT + DPO birikimli)
model.save_pretrained("worldcup_lora")
tokenizer.save_pretrained("worldcup_lora")

# Drive'a da yedekle — Colab kapanırsa kaybolmasın
model.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/worldcup_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/worldcup_lora")

# 2) Eğitim modelini bellekten at, GPU'yu boşalt
#    (restart sonrası 'trainer' tanımsız olabilir → güvenli silme)
for _v in ["model", "trainer", "dpo_trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# 3) fp16 tabanı CPU'ya yükle + adaptörü uygula + gerçekten merge et
base = "unsloth/gemma-3-12b-it"          # 4-bit DEĞİL, tam fp16 taban
m = AutoModelForImageTextToText.from_pretrained(
    base, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True,
)
m = PeftModel.from_pretrained(m, "worldcup_lora")
m = m.merge_and_unload()
tok = AutoTokenizer.from_pretrained(base)

# 4) Merge edilmiş tam modeli kendi repona push et
m.push_to_hub("faatihucar/worldcup-turkiye-gemma3-12b", token=HF_TOKEN, max_shard_size="5GB")
tok.push_to_hub("faatihucar/worldcup-turkiye-gemma3-12b", token=HF_TOKEN)

print("Bitti")


Unsloth: Restored added_tokens_decoder metadata in worldcup_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in worldcup_lora.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/worldcup_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/worldcup_lora.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mptkhrrdnp/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

Bitti


# **Top-Up ile modeli geliştirmek**

Top-up verisi ile modeli geliştirmek eğer ki top-up yapacaksanız yukarıdaki huggingface-push işlemini top-up yaptıktan sonra yapınız.

## **Top-UP eğitimi için Eğitilen LoRA Matrisini Yükleme**

In [4]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/World Cup fine-tuning/worldcup_lora",  # final adapter (CPT+SFT+DPO)
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Gemma3 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/258k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(4096, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                )
 

## **Top-up verisini yükleme**




In [5]:
from datasets import load_dataset
topup = load_dataset("faatihucar/worldcup-finetuning",
                     data_files="topup_mixed.jsonl", split="train", token=HF_TOKEN)

README.md:   0%|          | 0.00/223 [00:00<?, ?B/s]

topup_mixed.jsonl:   0%|          | 0.00/345k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
topup[5]

{'messages': [{'role': 'user',
   'content': 'Türk taraftarları neden her milli maçta statlarda benzer bir atmosfer yaratıyor, bunun sebebi nedir?'},
  {'role': 'assistant',
   'content': 'Bu, Türk milletinin futbol sevgisinden, milli kimlik duygusundan ve başarıya olan inancından kaynaklanıyor. Her maç, ulusal bir dayanışma ve birliktelik fırsatı olarak görülüyor.'}]}

## **Top-up verisini formatlama**

In [8]:
def formatting_sft(examples):
    return {"prompt": [tokenizer.apply_chat_template(m, tokenize=False,
            add_generation_prompt=False) for m in examples["messages"]]}
topup = topup.map(formatting_sft, batched=True)

def formatting_func(example):
    p = example["prompt"]
    return p if isinstance(p, list) else [p]

Map:   0%|          | 0/902 [00:00<?, ? examples/s]

## **Top-up SFT Trainer Oluşturma**

In [9]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = topup,
    dataset_text_field = "prompt",
    formatting_func = formatting_func,
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 4,     # A100
        gradient_accumulation_steps = 2,
        completion_only_loss = False,

        warmup_steps = 10,
        num_train_epochs = 1,                # KISA tut — 1 epoch
        # max_steps = 60,                    # çok küçük setse bunu kullan

        learning_rate = 1e-5,                # DÜŞÜK (modeli bozmamak için)
        embedding_learning_rate = 5e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/World Cup fine-tuning/TOPUP",
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        report_to = "none",
    ),
)
trainer.train()


/content/unsloth_compiled_cache/UnslothSFTTrainer.py:646: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  super().__init__(


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/902 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 902 | Num Epochs = 1 | Total steps = 113
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 523,763,712 of 12,711,088,752 (4.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


...

Step,Training Loss
1,1.206960
2,1.130094
3,1.182346
4,1.106661
5,0.996410
6,1.079153
7,1.163336
8,1.145789
9,0.907115
10,1.194417


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-50/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-50.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-113/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/TOPUP/checkpoint-113.


TrainOutput(global_step=113, training_loss=0.8915086652325318, metrics={'train_runtime': 495.4089, 'train_samples_per_second': 1.821, 'train_steps_per_second': 0.228, 'total_flos': 8322971412450240.0, 'train_loss': 0.8915086652325318, 'epoch': 1.0})

# **Modeli HuggingFaceye Pushlamak**

In [10]:
import torch, gc
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import PeftModel

# 1) Final adapter (CPT+SFT+DPO+TOPUP) — YENİ isimle kaydet
model.save_pretrained("worldcup_topup_lora")
tokenizer.save_pretrained("worldcup_topup_lora")
model.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/worldcup_topup_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/World Cup fine-tuning/worldcup_topup_lora")

# 2) Belleği boşalt (restart sonrası tanımsız isimleri atla)
for _v in ["model", "trainer", "dpo_trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# 3) fp16 tabanı yükle + AYNI adapter'ı uygula + merge et
base = "unsloth/gemma-3-12b-it"
m = AutoModelForImageTextToText.from_pretrained(
    base, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True,
)
m = PeftModel.from_pretrained(m, "worldcup_topup_lora")   # ← 1. adımla AYNI isim
m = m.merge_and_unload()
tok = AutoTokenizer.from_pretrained(base)

# 4) Yeni repoya push
m.push_to_hub("faatihucar/worldcup-turkiye-gemma3-12b", token=HF_TOKEN, max_shard_size="5GB")
tok.push_to_hub("faatihucar/worldcup-turkiye-gemma3-12b", token=HF_TOKEN)

print("Bitti")


Unsloth: Restored added_tokens_decoder metadata in worldcup_topup_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in worldcup_topup_lora.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/World Cup fine-tuning/worldcup_topup_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/World Cup fine-tuning/worldcup_topup_lora.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp2q3to4sc/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

Bitti


# **Bu fine-tune edilmiş modelini ollamaya yüklemek için GGUF formatına çevirmek gerek**

https://huggingface.co/spaces/ggml-org/gguf-my-repo

## **GGUF Modeli indirmek**

In [11]:
!wget "https://huggingface.co/faatihucar/worldcup-turkiye-gemma3-12b-Q4_K_M-GGUF/resolve/main/worldcup-turkiye-gemma3-12b-q4_k_m.gguf" -O "worldcup-turkiye-finetune-gemma3-12b-q4_k_m.gguf"


--2026-06-18 13:23:29--  https://huggingface.co/faatihucar/worldcup-turkiye-gemma3-12b-Q4_K_M-GGUF/resolve/main/worldcup-turkiye-gemma3-12b-q4_k_m.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.121, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/6a33f1014005b98559b20e80/9b5cb841acde564a65b19338638252e44adb962e0da37103ee5d4eaace793a4c?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27worldcup-turkiye-gemma3-12b-q4_k_m.gguf%3B+filename%3D%22worldcup-turkiye-gemma3-12b-q4_k_m.gguf%22%3B&user_id=public&Expires=1781792610&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmEzM2YxMDE0MDA1Yjk4NTU5YjIwZTgwLzliNWNiODQxYWNkZTU2NGE2NWIxOTMzODYzODI1MmU0NGFkYjk2MmUwZGEzNzEwM2VlNWQ0ZWFhY2U3OTNhNGNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LW

Model file oluştururken finetune ettiğim modelin template kısmına gir. Template yapıştır sonra modelin parametre kısmına gir

In [12]:
modelfile_content = '''FROM ./worldcup-turkiye-finetune-gemma3-12b-q4_k_m.gguf

TEMPLATE """{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if or (eq .Role "user") (eq .Role "system") }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{ if $last }}<start_of_turn>model
{{ end }}
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}{{ if not $last }}<end_of_turn>
{{ end }}
{{- end }}
{{- end }}"""

PARAMETER stop "<end_of_turn>"
PARAMETER temperature 1
PARAMETER top_k 64
PARAMETER top_p 0.95
'''

with open('Modelfile', 'w') as f:

    f.write(modelfile_content)

# **Ollama İndirme ve Key oluşturma**

In [ ]:
!sudo apt-get install zstd #Terminalde çalıştır

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (14.7 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently 

In [ ]:
#ollama indirme
!curl -fsSL https://ollama.com/install.sh | sh #Terminalde çalıştır

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
!ls -l

total 7137700
drwx------ 5 root root       4096 Jun 18 12:54 drive
-rw-r--r-- 1 root root        527 Jun 18 13:37 Modelfile
drwxr-xr-x 1 root root       4096 Jun  4 13:39 sample_data
drwxr-xr-x 4 root root       4096 Jun 18 12:55 unsloth_compiled_cache
drwxr-xr-x 2 root root       4096 Jun 18 13:09 worldcup_topup_lora
-rw-r--r-- 1 root root 7308979392 Jun 18 13:36 worldcup-turkiye-finetune-gemma3-12b-q4_k_m.gguf


In [14]:
!ollama serve

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIPjbttoULSKi6VvslKFkGMUX464LCFePm41LJaPuQ6pz

time=2026-06-18T13:37:29.115Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost h

Ollama Key al ve public keylere ekle sonrasında o eklenen key ile bu notebook bağlanacak

In [ ]:
!ollama create faatihucar/worldcup-turkiye-finetune-gemma3-12b #terminalde çalıştır

^C


In [ ]:
!ollama run faatihucar/worldcup-turkiye-finetune-gemma3-12b #terminalde çalıştır

In [ ]:
!ollama push faatihucar/worldcup-turkiye-finetune-gemma3-12b #terminalde çalıştır